# Step 2 — SageMaker Processing Job

Run `scripts/preprocessing.py` on a managed ml.m5.xlarge instance.

In [ ]:
import boto3
import sagemaker
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput

BUCKET = "YOUR-S3-BUCKET-NAME"
REGION = "us-east-2"
PREFIX = "symptom-data"

boto_session = boto3.Session(region_name=REGION)
session = sagemaker.Session(
    boto_session=boto_session,
    default_bucket=BUCKET  # Must specify — otherwise SageMaker uses its own default
)
role = sagemaker.get_execution_role()

print(f"Region: {session.boto_region_name}")
print(f"Role:   {role}")

In [ ]:
processor = ScriptProcessor(
    image_uri="763104351884.dkr.ecr.us-east-2.amazonaws.com/pytorch-training:2.0.0-cpu-py310",
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    command=["python3"],
    sagemaker_session=session,
)

processor.run(
    code="../scripts/preprocessing.py",
    inputs=[ProcessingInput(
        source=f"s3://{BUCKET}/{PREFIX}/raw/",
        destination="/opt/ml/processing/input"
    )],
    outputs=[ProcessingOutput(
        source="/opt/ml/processing/output",
        destination=f"s3://{BUCKET}/{PREFIX}/processed/"
    )],
    wait=True,
    logs=True
)
print("Processing Job complete ✅")

## Verify outputs in S3

In [ ]:
import boto3
s3 = boto3.client("s3", region_name=REGION)
response = s3.list_objects_v2(Bucket=BUCKET, Prefix=f"{PREFIX}/processed/")
for obj in response["Contents"]:
    print(f"{obj['Key']}  ({obj['Size']} bytes)")